# M12 · LAB · Modelos open weights en COLAB (transformers)

**KEEPCODING · AI Engineering · Clase 2 · Módulo M12 · Notebook 2 de 3**

Ejecutar **Gemma 4 E2B** (Google) y **Qwen 3.5 2B** (Alibaba) descargando los pesos con [transformers](https://huggingface.co/docs/transformers) y corriéndolos en la **GPU de Colab**. Sin API externa, sin Ollama: los pesos viven en tu sesión. Los mismos prompts del notebook 01, ejecutados sobre modelos open weights que tú mismo descargas.

## Recorrido

1. Setup · activar GPU + instalar transformers.
2. Descargar y cargar los dos modelos desde HuggingFace.
3. Comparativa · misma pregunta, dos modelos.
4. Casos de uso · resumir, extraer JSON, generar código.
5. Latencia y memoria · medir en la GPU de Colab.
6. Cuándo open weights propios vs API cloud.

## Por qué open weights en 2026

Modelos open weights de 2-9B parámetros tienen ya calidad workhorse-class y caben en una GPU modesta. Razones para descargarlos y servirlos tú:

- **Privacidad de datos** · los pesos y los prompts no salen de tu entorno.
- **Coste cero por token** · pagas cómputo (GPU), no por llamada.
- **Control total** · puedes cuantizar, fine-tunear o desplegar on-device.
- **Reproducibilidad** · fijas la versión exacta del modelo.

Trade-off · calidad inferior a frontier (GPT-5, Claude 4.7) en tareas complejas, y necesitas una GPU con VRAM suficiente.

> **Cómo seguir este lab**
>
> 1. **Lectura rápida** · pasa por los headers y mira qué tareas cubre. ~10 min.
> 2. **Ejecución** · necesitas **runtime con GPU** (la T4 gratis de Colab vale). La primera vez descarga ~9 GB de pesos (3-6 min). Después, cada respuesta tarda segundos. ~25 min en total.
> 3. **Modificar** · cambia los modelos por otras variantes (E4B, Qwen3.5-4B), prueba 4-bit, mide VRAM. Aquí se aprende de verdad.
>
> **Modelos:** `google/gemma-4-E2B-it` (~2.3B) y `Qwen/Qwen3.5-2B` (~2B). Ambos **Apache-2.0**, sin gating. Son multimodales pero aquí los usamos solo con texto.

## 1 · Setup · GPU + transformers

### 1.1 Activar la GPU en Colab

`Entorno de ejecución` → `Cambiar tipo de entorno` → **T4 GPU**. Sin GPU, cargar los modelos sería lentísimo (o se quedaría sin RAM).

In [ ]:
%pip install -qU transformers accelerate

> **Si Qwen 3.5 o Gemma 4 dan `KeyError` / "model type not recognized"**, tu `transformers` es anterior al soporte de estos modelos. Instala la versión de desarrollo y reinicia el runtime:
>
> ```python
> %pip install -qU "git+https://github.com/huggingface/transformers.git" accelerate
> ```

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No hay GPU. En Colab: Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU."
)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")

## 2 · Descargar y cargar los modelos

`from_pretrained` descarga los pesos de HuggingFace la primera vez (quedan cacheados en la sesión) y los carga en la GPU.

- **Gemma 4 E2B** se carga como un modelo de texto clásico (`AutoModelForCausalLM`).
- **Qwen 3.5** es multimodal: se carga con `AutoProcessor` + `AutoModelForImageTextToText` (la API oficial de su model card).

Usamos `dtype="auto"` (transformers elige el dtype nativo del modelo) y `device_map="auto"`. Juntos ocupan ~9 GB, así que entran en la T4 (16 GB).

> Ambos son Apache-2.0 y no requieren token. Si Gemma te pidiera aceptar términos, hazlo en su página de HuggingFace y ejecuta `from huggingface_hub import login; login()`.

In [ ]:
import time
from transformers import (
    pipeline,
    AutoProcessor,
    AutoModelForImageTextToText,
)

MODEL_GEMMA = "google/gemma-4-E2B-it"   # Google   · ~2.3B · Apache-2.0
MODEL_QWEN = "Qwen/Qwen3.5-2B"          # Alibaba  · ~2B   · Apache-2.0

# --- Gemma 4 E2B · pipeline de texto (la via mas simple, segun el blog de HF) ---
print(f"Cargando {MODEL_GEMMA} (descarga ~5 GB la primera vez)...")
gemma_pipe = pipeline(
    "text-generation",
    model=MODEL_GEMMA,
    dtype="auto",
    device_map="auto",
)

# --- Qwen 3.5 · processor + modelo image-text-to-text (solo usamos texto) ---
print(f"Cargando {MODEL_QWEN} (descarga ~4 GB la primera vez)...")
qwen_proc = AutoProcessor.from_pretrained(MODEL_QWEN)
qwen = AutoModelForImageTextToText.from_pretrained(
    MODEL_QWEN,
    dtype="auto",
    device_map="auto",
)

print("\nModelos cargados.")
print(f"VRAM en uso: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

### 2.1 Helper `chat(...)` · misma firma que en el notebook 01

Un único `chat(prompt, model=...)` que enruta al modelo correcto. Greedy (`do_sample=False`) para respuestas deterministas, como `temperature=0`. En Qwen desactivamos el *thinking mode* para que conteste directo (comparación justa con Gemma).

In [ ]:
def _gen_gemma(prompt: str, max_new_tokens: int) -> str:
    out = gemma_pipe(
        [{"role": "user", "content": prompt}],
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    return out[0]["generated_text"].strip()


def _gen_qwen(prompt: str, max_new_tokens: int) -> str:
    inputs = qwen_proc.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        chat_template_kwargs={"enable_thinking": False},  # modo directo, sin razonar
    ).to(qwen.device)
    with torch.no_grad():
        out = qwen.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    gen = out[0][inputs["input_ids"].shape[-1]:]
    return qwen_proc.decode(gen, skip_special_tokens=True).strip()


def chat(prompt: str, model: str = MODEL_GEMMA, max_new_tokens: int = 320) -> str:
    """Mismo signature que en el notebook 01. Enruta al modelo cargado."""
    if model == MODEL_GEMMA:
        return _gen_gemma(prompt, max_new_tokens)
    return _gen_qwen(prompt, max_new_tokens)


print("Helper listo. Smoke test:")
print(chat("Responde solo con la palabra: OK", model=MODEL_GEMMA))

## 3 · Comparativa · misma pregunta, dos modelos

La primera respuesta de cada modelo incluye el tiempo de calentamiento (carga de kernels CUDA); las siguientes van más rápidas.

In [ ]:
pregunta = "Cuéntame, en menos de 100 palabras, qué es el efecto Doppler y para qué se usa en astronomía moderna."

print("=== Gemma 4 E2B ===")
print(chat(pregunta, model=MODEL_GEMMA))
print()
print("=== Qwen 3.5 2B ===")
print(chat(pregunta, model=MODEL_QWEN))

**Observaciones esperadas:**

- Ambos modelos pequeños dan una explicación correcta de nivel divulgativo.
- A 2B parámetros pueden cometer imprecisiones en los detalles (corrimiento al rojo, exoplanetas). Para divulgación general es suficiente; para precisión científica, sube de tamaño o usa frontier.

## 4 · Casos de uso

### 4.1 Resumir un texto

In [ ]:
texto = """
El protocolo MCP (Model Context Protocol), creado por Anthropic en 2024, estandariza cómo
los LLM acceden a datos y tools externos. Un servidor MCP expone recursos (archivos, APIs,
bases de datos) bajo un schema común, y cualquier cliente MCP-compatible (Claude Desktop,
Cursor, Zed, Continue) puede consumirlos sin reescribir integraciones. Ha sido adoptado en
2026 como el equivalente de USB-C para integraciones LLM.
"""

prompt = f"Resume este texto en 2 frases para un lector técnico:\n\n{texto}"

print("=== Gemma 4 E2B ===")
print(chat(prompt, model=MODEL_GEMMA))
print()
print("=== Qwen 3.5 2B ===")
print(chat(prompt, model=MODEL_QWEN))

### 4.2 Extraer información estructurada (JSON)

In [ ]:
review = """
Compré el portátil hace 3 semanas. El teclado es excelente y la pantalla 4K se ve genial.
La batería dura 8 horas, justo lo prometido. Pero el ventilador hace ruido bajo carga,
y el trackpad detecta gestos falsos a veces. Lo recomendaría a un programador, no a un
diseñador. Precio 1800 EUR.
"""

prompt = f"""
Extrae de la siguiente review los aspectos positivos y negativos. Devuelve JSON con esta estructura:
{{
  "positivos": ["...", "..."],
  "negativos": ["...", "..."],
  "recomendado_para": "...",
  "precio_eur": 0
}}

Solo devuelve el JSON, sin texto adicional.

Review:
{review}
"""

print("=== Gemma 4 E2B ===")
print(chat(prompt, model=MODEL_GEMMA))
print()
print("=== Qwen 3.5 2B ===")
print(chat(prompt, model=MODEL_QWEN))

**Observación clave:** los modelos open weights pequeños a veces envuelven el JSON en texto o ```json ... ```. En producción se fuerza el formato con `response_format` / *grammars* (vLLM, llama.cpp) o se parsea con tolerancia. Lo viste en el notebook 01 (Structured Output).

### 4.3 Generar código

In [ ]:
prompt = """
Escribe una función Python que reciba una lista de strings y devuelva un dict con
el conteo de caracteres por string. Incluye docstring y type hints. No expliques nada,
solo el código.
"""

print("=== Gemma 4 E2B ===")
print(chat(prompt, model=MODEL_GEMMA))
print()
print("=== Qwen 3.5 2B ===")
print(chat(prompt, model=MODEL_QWEN))

**Observación:** Qwen lleva un fine-tuning fuerte de código y suele dar funciones más limpias. Para tareas de programación reales en local, los modelos de la familia Qwen Coder son la referencia open weights.

## 5 · Medir latencia y memoria

In [ ]:
import time

prompt = "Lista las 5 capitales europeas con más habitantes."

for nombre, modelo in [("Gemma 4 E2B", MODEL_GEMMA), ("Qwen 3.5 2B", MODEL_QWEN)]:
    t0 = time.time()
    respuesta = chat(prompt, model=modelo)
    elapsed = time.time() - t0
    print(f"{nombre}: {elapsed:.2f}s · {len(respuesta)} chars")

print(f"\nVRAM en uso con los dos modelos cargados: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

**Comparativa típica en Colab T4 (16 GB):**

| Modelo | Latencia (respuesta corta) | VRAM | Coste |
|---|---|---|---|
| Gemma 4 E2B | ~2-5 s | ~5 GB | $0 (Colab Free) |
| Qwen 3.5 2B | ~2-6 s | ~4 GB | $0 (Colab Free) |

El coste real es **cómputo**, no tokens: en Colab Free es gratis (con límites de sesión); en tu propia GPU pagas electricidad y hardware. La primera ejecución suma el tiempo de descarga (~9 GB una vez por sesión).

> **¿Sin VRAM suficiente?** Carga los modelos de uno en uno (`del modelo; import gc; gc.collect(); torch.cuda.empty_cache()` entre uno y otro), usa las variantes más pequeñas, o carga en 4-bit con `bitsandbytes` (`load_in_4bit=True`).

## 6 · Cuándo open weights propios vs API cloud

**Usa open weights que descargas tú** cuando:

- Los datos no pueden salir de tu entorno (privacidad, compliance).
- Tienes volumen alto y predecible: el coste por token de la API se dispara.
- Quieres fine-tunear, cuantizar o desplegar on-device / edge.
- Necesitas reproducibilidad exacta (versión fija del modelo).

**Usa API cloud (GPT-5, Claude 4.7)** cuando:

- Necesitas la máxima capacidad en tareas difíciles (un 2B no escribe un paper).
- No quieres gestionar GPUs ni infra.
- El volumen es bajo o irregular y el pago por uso compensa.

## Recap

- Descargaste **Gemma 4 E2B** y **Qwen 3.5 2B** con `transformers` y los corriste en la **GPU de Colab**, sin Ollama ni API externa.
- Mismos prompts del notebook 01: comparación, resumen, JSON, código.
- `chat(prompt, model=...)` con greedy = `temperature=0`; en Qwen, *thinking mode* desactivado para respuestas directas.
- Trade-off claro: cloud gana en capacidad, open weights propios ganan en coste/privacidad/control.
- En producción 2026: router de modelos (lo viste en el deck, slide 11). Tareas simples → modelo pequeño local; tareas difíciles → frontier cloud.

### Retos para practicar

1. Cambia a `google/gemma-4-E4B-it` y `Qwen/Qwen3.5-4B` y compara calidad vs latencia/VRAM.
2. Carga Qwen en 4-bit (`bitsandbytes`) y mide cuánta VRAM ahorras.
3. Activa el *thinking mode* de Qwen (`enable_thinking=True`) en el puzzle del notebook 03 y compara con el modo directo.